## measles cases around the world by tidyteusday

In [1]:
# import the  libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np      
import os
import warnings
warnings.filterwarnings('ignore')

# Add Plotly for interactive visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# load the  yearly data
yearly_measles = pd.read_csv('../data/cases_year.csv')  
yearly_measles.head()

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,measles_clinical,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population
0,AFRO,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,53,1.46,13,13,0,0,0.35,8.0,0.02
1,AFRO,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,0,0.00,29,29,0,0,0.75,56.0,0.15
2,AFRO,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,0,0.00,3,3,0,0,0.08,46.0,0.12
3,AFRO,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,0,1.55,2,2,0,0,0.05,45.0,0.11
4,AFRO,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,1,1.20,11,11,0,0,0.27,33.0,0.08


| Variable | Class | Description |
|---|---|---|
| region | character | Region name |
| country | character | Country name |
| iso3 | character | Three-letter country code |
| year | character | Year |
| total_population | character | Country population |
| annualized_population_most_recent_year_only | character | Annualized population (2025) |
| total_suspected_measles_rubella_cases | character | Suspected measles/rubella cases: a suspected case is one in which a patient has fever and maculopapular (non-vesicular) rash, or in whom a health-care worker suspects measles (or rubella). |
| measles_total | character | Total measles cases: sum of clinically compatible, epidemiologically linked, and laboratory-confirmed cases. |
| measles_lab_confirmed | character | Laboratory-confirmed measles cases: a suspected measles case confirmed positive by testing in a proficient laboratory, with vaccine-associated illness ruled out. |
| measles_epi_linked | character | Epidemiologically linked measles cases: a suspected measles case not laboratory-confirmed, but geographically and temporally related (rash onset 7–23 days apart) to a laboratory-confirmed or another epidemiologically linked measles case. |
| measles_clinical | character | Clinically compatible measles cases: a suspected case with fever and maculopapular (non-vesicular) rash and at least one of cough, coryza, or conjunctivitis, with no adequate specimen and no epidemiological link to a laboratory-confirmed measles case or other communicable disease. |
| measles_incidence_rate_per_1000000_total_population | character | Measles cases per million population |
| rubella_total | character | Total rubella cases |
| rubella_lab_confirmed | character | Laboratory-confirmed rubella cases |
| rubella_epi_linked | character | Epidemiologically linked rubella cases |
| rubella_clinical | character | Clinically compatible rubella cases |
| rubella_incidence_rate_per_1000000_total_population | character | Rubella cases per million population |
| discarded_cases | character | Discarded cases: a suspected case investigated and discarded as non-measles (and non-rubella). |
| discarded_non_measles_rubella_cases_per_100000_total_population | character | Discarded cases per million population |

In [3]:
yearly_measles.info()

<class 'pandas.DataFrame'>
RangeIndex: 2382 entries, 0 to 2381
Data columns (total 19 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   region                                                           2382 non-null   str    
 1   country                                                          2380 non-null   str    
 2   iso3                                                             2382 non-null   str    
 3   year                                                             2382 non-null   int64  
 4   total_population                                                 2382 non-null   int64  
 5   annualized_population_most_recent_year_only                      2382 non-null   int64  
 6   total_suspected_measles_rubella_cases                            2295 non-null   float64
 7   measles_total                                        

In [4]:
# check for duplicates
yearly_measles.duplicated().sum()

np.int64(0)

## cleaning the data

In [5]:
# standardize the column names 
yearly_measles.columns = yearly_measles.columns.str.strip().str.lower().str.replace(' ', '_')
yearly_measles.head()

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,measles_clinical,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population
0,AFRO,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,53,1.46,13,13,0,0,0.35,8.0,0.02
1,AFRO,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,0,0.00,29,29,0,0,0.75,56.0,0.15
2,AFRO,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,0,0.00,3,3,0,0,0.08,46.0,0.12
3,AFRO,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,0,1.55,2,2,0,0,0.05,45.0,0.11
4,AFRO,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,1,1.20,11,11,0,0,0.27,33.0,0.08


In [6]:
#normalize region codes 
region_map = {
    "AFRO": "AFR", "EURO": "EUR", "AMRO": "AMR",
    "EMRO": "EMR", "SEARO": "SEAR", "WPRO": "WPR"
}
yearly_measles["region"] = yearly_measles["region"].replace(region_map)

In [7]:
#separate menaingfull nulls from the rest
surveillance_cols = [
    "measles_lab_confirmed", "measles_epi_linked", "measles_clinical",
    "rubella_lab_confirmed", "rubella_epi_linked", "rubella_clinical",
    "discarded_cases"
]
yearly_measles["surveillance_reported"] = yearly_measles[surveillance_cols].notna().all(axis=1)
# print head 
yearly_measles.head()

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,measles_clinical,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population,surveillance_reported
0,AFR,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,53,1.46,13,13,0,0,0.35,8.0,0.02,True
1,AFR,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,0,0.00,29,29,0,0,0.75,56.0,0.15,True
2,AFR,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,0,0.00,3,3,0,0,0.08,46.0,0.12,True
3,AFR,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,0,1.55,2,2,0,0,0.05,45.0,0.11,True
4,AFR,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,1,1.20,11,11,0,0,0.27,33.0,0.08,True


In [8]:
# create a new column to identify rows where measles cases are reported but lab confirmation is missing
yearly_measles["measles_breakdown_missing"] = (
    yearly_measles["measles_total"] > 0
) & (yearly_measles["measles_lab_confirmed"].isna())
yearly_measles.head()  

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,...,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population,surveillance_reported,measles_breakdown_missing
0,AFR,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,...,1.46,13,13,0,0,0.35,8.0,0.02,True,False
1,AFR,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,...,0.00,29,29,0,0,0.75,56.0,0.15,True,False
2,AFR,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,...,0.00,3,3,0,0,0.08,46.0,0.12,True,False
3,AFR,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,...,1.55,2,2,0,0,0.05,45.0,0.11,True,False
4,AFR,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,...,1.20,11,11,0,0,0.27,33.0,0.08,True,False


In [9]:
# Validate: suspected >= measles_total + rubella_total
yearly_measles["suspect_sum"] = yearly_measles["measles_total"] + yearly_measles["rubella_total"]
yearly_measles["suspect_mismatch"] = yearly_measles["total_suspected_measles_rubella_cases"] < yearly_measles["suspect_sum"]
 
print("\n── Data quality flags ──")
print(f"Rows missing surveillance breakdown: {(~yearly_measles['surveillance_reported']).sum()}")
print(f"Rows with measles but no breakdown:  {yearly_measles['measles_breakdown_missing'].sum()}")
print(f"Rows where suspect < total cases:    {yearly_measles['suspect_mismatch'].sum()}")
 


── Data quality flags ──
Rows missing surveillance breakdown: 87
Rows with measles but no breakdown:  0
Rows where suspect < total cases:    13


In [10]:
# Convert data types
numeric_cols = yearly_measles.select_dtypes(include=['object']).columns.difference(['region', 'country', 'iso3'])
for col in numeric_cols:
    yearly_measles[col] = pd.to_numeric(yearly_measles[col], errors='coerce')

yearly_measles['year'] = yearly_measles['year'].astype(int)

# Check data types after conversion
print("\n── Data Types After Conversion ──")
print(yearly_measles.dtypes)
print(f"\nMissing values per column:")
print(yearly_measles.isnull().sum())


── Data Types After Conversion ──
region                                                                 str
country                                                                str
iso3                                                                   str
year                                                                 int64
total_population                                                     int64
annualized_population_most_recent_year_only                          int64
total_suspected_measles_rubella_cases                              float64
measles_total                                                        int64
measles_lab_confirmed                                                int64
measles_epi_linked                                                   int64
measles_clinical                                                     int64
measles_incidence_rate_per_1000000_total_population                float64
rubella_total                                                    

In [11]:
# Fill missing values for surveillance columns with 0 (indicating unreported)
yearly_measles[surveillance_cols] = yearly_measles[surveillance_cols].fillna(0)

# For population fields, forward-fill then backward-fill
yearly_measles['total_population'] = yearly_measles.groupby('country')['total_population'].transform(lambda x: x.ffill().bfill())

print("\n── Missing values after filling ──")
print(yearly_measles.isnull().sum())


── Missing values after filling ──
region                                                               0
country                                                              2
iso3                                                                 0
year                                                                 0
total_population                                                     2
annualized_population_most_recent_year_only                          0
total_suspected_measles_rubella_cases                               87
measles_total                                                        0
measles_lab_confirmed                                                0
measles_epi_linked                                                   0
measles_clinical                                                     0
measles_incidence_rate_per_1000000_total_population                  0
rubella_total                                                        0
rubella_lab_confirmed                    

In [12]:
# Remove duplicates
initial_rows = len(yearly_measles)
yearly_measles = yearly_measles.drop_duplicates(subset=['region', 'country', 'iso3', 'year'])
removed_rows = initial_rows - len(yearly_measles)
print(f"\n── Duplicates Removed ──")
print(f"Removed {removed_rows} duplicate rows")

# Rename cleaned indicator columns for clarity
yearly_measles = yearly_measles.drop(columns=['suspect_sum', 'suspect_mismatch'])

print(f"\n── Final Dataset Summary ──")
print(f"Shape: {yearly_measles.shape}")
print(f"Rows: {yearly_measles.shape[0]}, Columns: {yearly_measles.shape[1]}")


── Duplicates Removed ──
Removed 0 duplicate rows

── Final Dataset Summary ──
Shape: (2382, 21)
Rows: 2382, Columns: 21


In [13]:
# Save cleaned data
output_path = '../data/cleaned/cases_year_clean.csv'
yearly_measles.to_csv(output_path, index=False)
print(f"\n✓ Cleaned yearly data saved to: {output_path}")

# Final validation
print(f"\n── Final Validation ──")
cleaned_data = pd.read_csv(output_path)
print(f"Loaded cleaned data shape: {cleaned_data.shape}")
print(f"\nColumn names:")
print(cleaned_data.columns.tolist())


✓ Cleaned yearly data saved to: ../data/cleaned/cases_year_clean.csv

── Final Validation ──
Loaded cleaned data shape: (2382, 21)

Column names:
['region', 'country', 'iso3', 'year', 'total_population', 'annualized_population_most_recent_year_only', 'total_suspected_measles_rubella_cases', 'measles_total', 'measles_lab_confirmed', 'measles_epi_linked', 'measles_clinical', 'measles_incidence_rate_per_1000000_total_population', 'rubella_total', 'rubella_lab_confirmed', 'rubella_epi_linked', 'rubella_clinical', 'rubella_incidence_rate_per_1000000_total_population', 'discarded_cases', 'discarded_non_measles_rubella_cases_per_100000_total_population', 'surveillance_reported', 'measles_breakdown_missing']


## Exploratory Data Analysis (EDA)

In [14]:
# Descriptive statistics
print("═" * 80)
print("DESCRIPTIVE STATISTICS")
print("═" * 80)
print(yearly_measles.describe())
print("\n")
print(yearly_measles[['measles_total', 'rubella_total', 'measles_lab_confirmed']].describe())

════════════════════════════════════════════════════════════════════════════════
DESCRIPTIVE STATISTICS
════════════════════════════════════════════════════════════════════════════════
              year  total_population  \
count  2382.000000      2.380000e+03   
mean   2018.631402      4.272505e+07   
std       3.958222      1.507132e+08   
min    2012.000000      1.757000e+03   
25%    2015.000000      2.878645e+06   
50%    2019.000000      1.009075e+07   
75%    2022.000000      3.172827e+07   
max    2025.000000      1.463866e+09   

       annualized_population_most_recent_year_only  \
count                                 2.382000e+03   
mean                                  4.074782e+07   
std                                   1.456164e+08   
min                                   7.590000e+02   
25%                                   2.797780e+06   
50%                                   9.620074e+06   
75%                                   3.021081e+07   
max                   

In [15]:
# Cases by region
print("\n" + "═" * 80)
print("MEASLES CASES BY REGION")
print("═" * 80)
region_analysis = yearly_measles.groupby('region').agg({
    'measles_total': ['sum', 'mean', 'max'],
    'rubella_total': ['sum', 'mean'],
    'country': 'nunique',
    'year': 'nunique'
}).round(2)
region_analysis.columns = ['_'.join(col).strip() for col in region_analysis.columns.values]
print(region_analysis)



════════════════════════════════════════════════════════════════════════════════
MEASLES CASES BY REGION
════════════════════════════════════════════════════════════════════════════════
        measles_total_sum  measles_total_mean  measles_total_max  \
region                                                             
AFR                901018             1499.20             213291   
AMR                 58733              179.61              20901   
EMR                517334             1808.86              49571   
EUR                548694              809.28              57332   
SEAR               632557             4245.35              82461   
WPR                434938             1275.48              53906   

        rubella_total_sum  rubella_total_mean  country_nunique  year_nunique  
region                                                                        
AFR                 67690              112.63               46            14  
AMR                     5      

In [16]:
# Temporal trends
print("\n" + "═" * 80)
print("MEASLES CASES OVER TIME")
print("═" * 80)
yearly_trend = yearly_measles.groupby('year').agg({
    'measles_total': 'sum',
    'rubella_total': 'sum',
    'measles_lab_confirmed': 'sum',
    'country': 'nunique'
}).round(0)
yearly_trend.columns = ['Total_Measles', 'Total_Rubella', 'Lab_Confirmed', 'Countries_Reporting']
print(yearly_trend)



════════════════════════════════════════════════════════════════════════════════
MEASLES CASES OVER TIME
════════════════════════════════════════════════════════════════════════════════
      Total_Measles  Total_Rubella  Lab_Confirmed  Countries_Reporting
year                                                                  
2012         112548          61382          46919                  149
2013         178526          67433          76271                  147
2014         290888          35207         101439                  154
2015         247474          22418          70653                  176
2016         180015          18679          46164                  179
2017         168190          10712          43913                  182
2018         276157          17045          90139                  182
2019         541401          47766         118203                  187
2020          93840           7501          36899                  175
2021          59619           74

In [17]:
# Correlation analysis
print("\n" + "═" * 80)
print("CORRELATION ANALYSIS")
print("═" * 80)
correlation_cols = ['measles_total', 'measles_lab_confirmed', 'measles_incidence_rate_per_1000000_total_population',
                    'rubella_total', 'rubella_lab_confirmed', 'total_population']
corr_matrix = yearly_measles[correlation_cols].corr()
print(corr_matrix.round(3))



════════════════════════════════════════════════════════════════════════════════
CORRELATION ANALYSIS
════════════════════════════════════════════════════════════════════════════════
                                                    measles_total  \
measles_total                                               1.000   
measles_lab_confirmed                                       0.464   
measles_incidence_rate_per_1000000_total_popula...          0.499   
rubella_total                                               0.155   
rubella_lab_confirmed                                       0.074   
total_population                                            0.386   

                                                    measles_lab_confirmed  \
measles_total                                                       0.464   
measles_lab_confirmed                                               1.000   
measles_incidence_rate_per_1000000_total_popula...                  0.114   
rubella_total           

In [18]:
# Visualization: Cases by region (Plotly)
region_cases = yearly_measles.groupby('region')['measles_total'].sum().sort_values(ascending=False)
fig = px.bar(x=region_cases.index, y=region_cases.values, 
             labels={'x': 'Region', 'y': 'Number of Cases'},
             title='Total Measles Cases by Region (2012-2025)',
             color=region_cases.values,
             color_continuous_scale='Viridis')
fig.update_layout(showlegend=False, height=600, hovermode='x unified')
fig.show()

In [19]:
# Visualization: Correlation heatmap (Plotly)
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont={"size": 10},
    colorbar=dict(title='Correlation')
))
fig.update_layout(title='Correlation Matrix of Key Variables',
                  height=700, width=900)
fig.show()

In [20]:
# Visualization: Lab-confirmed vs clinical cases (Plotly)
measles_breakdown = yearly_measles.groupby('year')[['measles_lab_confirmed', 'measles_epi_linked', 'measles_clinical']].sum().reset_index()
fig = go.Figure()
fig.add_trace(go.Bar(x=measles_breakdown['year'], y=measles_breakdown['measles_lab_confirmed'],
                     name='Lab Confirmed', marker_color='#1f77b4'))
fig.add_trace(go.Bar(x=measles_breakdown['year'], y=measles_breakdown['measles_epi_linked'],
                     name='Epidemiologically Linked', marker_color='#ff7f0e'))
fig.add_trace(go.Bar(x=measles_breakdown['year'], y=measles_breakdown['measles_clinical'],
                     name='Clinical', marker_color='#2ca02c'))
fig.update_layout(title='Measles Cases by Classification Type Over Time',
                  xaxis_title='Year', yaxis_title='Number of Cases',
                  barmode='group', height=600, hovermode='x unified')
fig.show()

In [21]:
# Visualization: Distribution of measles incidence rates (Plotly)
fig = make_subplots(rows=1, cols=2,
                     subplot_titles=('Distribution of Measles Incidence Rate', 
                                     'Measles Incidence Rate by Region'))

# Histogram
fig.add_trace(
    go.Histogram(x=yearly_measles['measles_incidence_rate_per_1000000_total_population'],
                 nbinsx=50, name='Histogram', marker_color='steelblue'),
    row=1, col=1
)

# Boxplot by region
for region in yearly_measles['region'].unique():
    region_data = yearly_measles[yearly_measles['region'] == region]['measles_incidence_rate_per_1000000_total_population']
    fig.add_trace(
        go.Box(y=region_data, name=region),
        row=1, col=2
    )

fig.update_xaxes(title_text='Incidence Rate (per 1,000,000)', row=1, col=1)
fig.update_xaxes(title_text='Region', row=1, col=2)
fig.update_yaxes(title_text='Frequency', row=1, col=1)
fig.update_yaxes(title_text='Incidence Rate (per 1,000,000)', row=1, col=2)
fig.update_layout(height=500, showlegend=False)
fig.show()

In [22]:
# Visualization: Measles cases over time (Plotly)
yearly_trend_plot = yearly_measles.groupby('year')[['measles_total', 'rubella_total']].sum().reset_index()
fig = go.Figure()
fig.add_trace(go.Scatter(x=yearly_trend_plot['year'], y=yearly_trend_plot['measles_total'],
                         mode='lines+markers', name='Measles Total',
                         line=dict(color='#1f77b4', width=3), marker=dict(size=8)))
fig.add_trace(go.Scatter(x=yearly_trend_plot['year'], y=yearly_trend_plot['rubella_total'],
                         mode='lines+markers', name='Rubella Total',
                         line=dict(color='#ff7f0e', width=3), marker=dict(size=8)))
fig.update_layout(title='Measles and Rubella Cases Over Time (2012-2025)',
                  xaxis_title='Year', yaxis_title='Number of Cases',
                  height=600, hovermode='x unified')
fig.show()

In [23]:
# Data quality summary
print("\n" + "═" * 80)
print("DATA QUALITY SUMMARY")
print("═" * 80)
print(f"Total records: {len(yearly_measles):,}")
print(f"Date range: {yearly_measles['year'].min():.0f} - {yearly_measles['year'].max():.0f}")
print(f"Number of regions: {yearly_measles['region'].nunique()}")
print(f"Number of countries: {yearly_measles['country'].nunique()}")
print(f"\nRecords with surveillance data: {yearly_measles['surveillance_reported'].sum():,} ({yearly_measles['surveillance_reported'].sum()/len(yearly_measles)*100:.1f}%)")
print(f"Records with measles breakdown missing: {yearly_measles['measles_breakdown_missing'].sum():,}")



════════════════════════════════════════════════════════════════════════════════
DATA QUALITY SUMMARY
════════════════════════════════════════════════════════════════════════════════
Total records: 2,382
Date range: 2012 - 2025
Number of regions: 6
Number of countries: 193

Records with surveillance data: 2,295 (96.3%)
Records with measles breakdown missing: 0


In [24]:
# Top 15 countries by total measles cases
print("\n" + "═" * 80)
print("TOP 15 COUNTRIES BY TOTAL MEASLES CASES (2012-2025)")
print("═" * 80)
top_countries = yearly_measles.groupby('country').agg({
    'measles_total': 'sum',
    'measles_lab_confirmed': 'sum',
    'rubella_total': 'sum',
    'year': 'nunique'
}).sort_values('measles_total', ascending=False).head(15).round(0)
top_countries.columns = ['Total Measles', 'Lab Confirmed', 'Total Rubella', 'Years Reported']
print(top_countries)



════════════════════════════════════════════════════════════════════════════════
TOP 15 COUNTRIES BY TOTAL MEASLES CASES (2012-2025)
════════════════════════════════════════════════════════════════════════════════
                                  Total Measles  Lab Confirmed  Total Rubella  \
country                                                                         
India                                    474525          78576          33057   
Madagascar                               225947           1745           2108   
Nigeria                                  210073          29000           7363   
China                                    172872         162159         125908   
Philippines                              148930          28448           2160   
Yemen                                    147829           9428          10784   
Ukraine                                  136311          21573           1409   
Pakistan                                 132292         

In [25]:
# Countries that drove the 2019 measles spike (vs 2018)
country_year = (
    yearly_measles
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

pivot_cases = country_year.pivot(index='country', columns='year', values='measles_total').fillna(0)
pivot_cases['increase_2019_vs_2018'] = pivot_cases.get(2019, 0) - pivot_cases.get(2018, 0)

spike_drivers = (
    pivot_cases[pivot_cases['increase_2019_vs_2018'] > 0][['increase_2019_vs_2018']]
    .sort_values('increase_2019_vs_2018', ascending=False)
)

global_spike = spike_drivers['increase_2019_vs_2018'].sum()
top_n = 15
top_spike_drivers = spike_drivers.head(top_n).copy()
top_spike_drivers['share_of_global_spike_pct'] = (
    top_spike_drivers['increase_2019_vs_2018'] / global_spike * 100
).round(2)

print('Countries driving the 2019 spike (increase from 2018 to 2019):')
print(top_spike_drivers)
print(f"\nTotal global increase (2019 vs 2018): {int(global_spike):,}")
print(
    f"Top {top_n} countries explain "
    f"{top_spike_drivers['share_of_global_spike_pct'].sum():.2f}% of that increase."
)

# Plotly horizontal bar chart
plot_df = top_spike_drivers.sort_values('increase_2019_vs_2018', ascending=True).reset_index()
fig = px.bar(plot_df, y='country', x='increase_2019_vs_2018',
             orientation='h',
             title='Top Countries Driving the 2019 Measles Spike (vs 2018)',
             labels={'increase_2019_vs_2018': 'Increase in Measles Cases', 'country': 'Country'},
             color='increase_2019_vs_2018',
             color_continuous_scale='Reds')
fig.update_layout(height=700, showlegend=False, hovermode='y unified')
fig.show()

Countries driving the 2019 spike (increase from 2018 to 2019):
year                              increase_2019_vs_2018  \
country                                                   
Madagascar                                     201239.0   
Philippines                                     25579.0   
Nigeria                                         21466.0   
Democratic Republic of the Congo                12975.0   
Kazakhstan                                      12750.0   
Brazil                                          10575.0   
Viet Nam                                         4955.0   
Tunisia                                          4650.0   
Ukraine                                          4114.0   
Myanmar                                          3856.0   
Bangladesh                                       3216.0   
Iraq                                             3130.0   
Angola                                           2689.0   
Algeria                                          229

In [26]:
# Countries that drove the 2023 to 2024 measles increase
country_year_23_24 = (
    yearly_measles
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

pivot_23_24 = country_year_23_24.pivot(index='country', columns='year', values='measles_total').fillna(0)
pivot_23_24['increase_2024_vs_2023'] = pivot_23_24.get(2024, 0) - pivot_23_24.get(2023, 0)

drivers_23_24 = (
    pivot_23_24[pivot_23_24['increase_2024_vs_2023'] > 0][['increase_2024_vs_2023']]
    .sort_values('increase_2024_vs_2023', ascending=False)
)

global_increase_23_24 = drivers_23_24['increase_2024_vs_2023'].sum()
top_n_23_24 = 15
top_drivers_23_24 = drivers_23_24.head(top_n_23_24).copy()
top_drivers_23_24['share_of_increase_pct'] = (
    top_drivers_23_24['increase_2024_vs_2023'] / global_increase_23_24 * 100
).round(2)

print('Countries driving the 2023 to 2024 increase:')
print(top_drivers_23_24)
print(f"\nTotal global increase (2024 vs 2023): {int(global_increase_23_24):,}")
print(
    f"Top {top_n_23_24} countries explain "
    f"{top_drivers_23_24['share_of_increase_pct'].sum():.2f}% of that increase."
)

# Plotly horizontal bar chart
plot_23_24 = top_drivers_23_24.sort_values('increase_2024_vs_2023', ascending=True).reset_index()
fig = px.bar(plot_23_24, y='country', x='increase_2024_vs_2023',
             orientation='h',
             title='Top Countries Driving Measles Increase (2023 to 2024)',
             labels={'increase_2024_vs_2023': 'Increase in Measles Cases', 'country': 'Country'},
             color='increase_2024_vs_2023',
             color_continuous_scale='Oranges')
fig.update_layout(height=700, showlegend=False, hovermode='y unified')
fig.show()

Countries driving the 2023 to 2024 increase:
year                                                increase_2024_vs_2023  \
country                                                                     
Romania                                                           27320.0   
Iraq                                                              22734.0   
Ethiopia                                                          13696.0   
Kazakhstan                                                        13036.0   
Russian Federation                                                 9116.0   
Thailand                                                           8155.0   
Afghanistan                                                        7240.0   
Kyrgyzstan                                                         6945.0   
Pakistan                                                           6748.0   
Burkina Faso                                                       5712.0   
Côte d'Ivoire                  

In [27]:
# Compare 2019 outbreak countries vs 2023 resurgence countries
comparison_df = pd.DataFrame({
    '2019_spike_rank': top_spike_drivers['increase_2019_vs_2018'],
    '2024_spike_rank': top_drivers_23_24['increase_2024_vs_2023']
}).fillna(0)

comparison_df['appears_in_both'] = (comparison_df['2019_spike_rank'] > 0) & (comparison_df['2024_spike_rank'] > 0)
comparison_df = comparison_df.sort_values('appears_in_both', ascending=False)

print("\n" + "═" * 80)
print("COMPARISON: 2019 OUTBREAK vs 2024 RESURGENCE")
print("═" * 80)
print(f"Countries in 2019 top drivers: {len(top_spike_drivers)}")
print(f"Countries in 2024 top drivers: {len(top_drivers_23_24)}")
print(f"Countries appearing in BOTH outbreaks: {comparison_df['appears_in_both'].sum()}")
print("\nCountries driving both outbreaks:")
print(comparison_df[comparison_df['appears_in_both']].sort_values('2019_spike_rank', ascending=False))

# Plotly scatter plot
comparison_reset = comparison_df.reset_index()
comparison_reset['color'] = comparison_reset['appears_in_both'].map({True: 'Both Outbreaks', False: 'Single Outbreak'})

fig = px.scatter(comparison_reset, x='2019_spike_rank', y='2024_spike_rank',
                 color='color',
                 color_discrete_map={'Both Outbreaks': '#d62728', 'Single Outbreak': '#1f77b4'},
                 size='2019_spike_rank',
                 hover_name='country',
                 title='Comparison: 2019 Outbreak vs 2024 Resurgence',
                 labels={'2019_spike_rank': 'Increase in Cases 2019 vs 2018',
                        '2024_spike_rank': 'Increase in Cases 2024 vs 2023'})

# Add text labels for both-outbreak countries
for idx, row in comparison_reset[comparison_reset['appears_in_both']].iterrows():
    fig.add_annotation(x=row['2019_spike_rank'], y=row['2024_spike_rank'],
                      text=row['country'], showarrow=False, font=dict(size=9))

fig.update_layout(height=700, hovermode='closest')
fig.show()


════════════════════════════════════════════════════════════════════════════════
COMPARISON: 2019 OUTBREAK vs 2024 RESURGENCE
════════════════════════════════════════════════════════════════════════════════
Countries in 2019 top drivers: 15
Countries in 2024 top drivers: 15
Countries appearing in BOTH outbreaks: 4

Countries driving both outbreaks:
            2019_spike_rank  2024_spike_rank  appears_in_both
country                                                      
Kazakhstan          12750.0          13036.0             True
Viet Nam             4955.0           2007.0             True
Iraq                 3130.0          22734.0             True
Ethiopia             2259.0          13696.0             True


In [28]:
median_incidence = (
    yearly_measles
    .groupby(["region", "year"])["measles_incidence_rate_per_1000000_total_population"]
    .median()
    .reset_index(name="median_incidence")
)

fig = px.line(
    median_incidence,
    x="year",
    y="median_incidence",
    color="region",
    markers=True,
    title="Median Measles Incidence Rate by Region Over Time",
    labels={
        "year": "Year",
        "median_incidence": "Median Incidence Rate (per 1,000,000)",
        "region": "Region"
    }
)

fig.update_layout(hovermode="x unified", height=600)
fig.show()

In [29]:
# EMR countries driving the 2023 spike (increase from 2022 to 2023)
emr_data = yearly_measles[yearly_measles['region'] == 'EMR'].copy()

emr_country_year = (
    emr_data
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

emr_pivot = emr_country_year.pivot(index='country', columns='year', values='measles_total').fillna(0)
emr_pivot['increase_2023_vs_2022'] = emr_pivot.get(2023, 0) - emr_pivot.get(2022, 0)

emr_spike_drivers = (
    emr_pivot[emr_pivot['increase_2023_vs_2022'] > 0][['increase_2023_vs_2022']]
    .sort_values('increase_2023_vs_2022', ascending=False)
)

emr_global_spike = emr_spike_drivers['increase_2023_vs_2022'].sum()
emr_top_n = 10
emr_top_drivers = emr_spike_drivers.head(emr_top_n).copy()
emr_top_drivers['share_of_emr_spike_pct'] = (
    emr_top_drivers['increase_2023_vs_2022'] / emr_global_spike * 100
).round(2)

print('EMR countries driving the 2023 spike (increase from 2022 to 2023):')
print(emr_top_drivers)
print(f"\nTotal EMR increase (2023 vs 2022): {int(emr_global_spike):,}")
print(
    f"Top {emr_top_n} countries explain "
    f"{emr_top_drivers['share_of_emr_spike_pct'].sum():.2f}% of the EMR increase."
)

# Interactive Plotly bar chart
emr_plot = emr_top_drivers.sort_values('increase_2023_vs_2022', ascending=True).reset_index()
fig = px.bar(
    emr_plot,
    y='country',
    x='increase_2023_vs_2022',
    orientation='h',
    color='increase_2023_vs_2022',
    color_continuous_scale='Tealgrn',
    title='Top EMR Countries Driving the 2023 Measles Spike (vs 2022)',
    labels={
        'increase_2023_vs_2022': 'Increase in Measles Cases',
        'country': 'Country'
    },
    hover_data={'share_of_emr_spike_pct': True}
)
fig.update_layout(height=600, showlegend=False)
fig.show()

EMR countries driving the 2023 spike (increase from 2022 to 2023):
year                        increase_2023_vs_2022  share_of_emr_spike_pct
country                                                                  
Yemen                                     25647.0                   49.63
Iraq                                       9630.0                   18.64
Pakistan                                   9598.0                   18.57
Sudan                                      2400.0                    4.64
Saudi Arabia                               2013.0                    3.90
Syrian Arab Republic                        527.0                    1.02
Iran (Islamic Republic of)                  414.0                    0.80
United Arab Emirates                        408.0                    0.79
Lebanon                                     260.0                    0.50
Egypt                                       242.0                    0.47

Total EMR increase (2023 vs 2022): 51,675
To

In [30]:
# Compare EMR vs other regions for 2022 to 2023 measles increase
region_year = (
    yearly_measles
    .groupby(['region', 'year'], as_index=False)['measles_total']
    .sum()
)

region_pivot = region_year.pivot(index='region', columns='year', values='measles_total').fillna(0)
region_pivot['increase_2023_vs_2022'] = region_pivot.get(2023, 0) - region_pivot.get(2022, 0)
region_compare = region_pivot[['increase_2023_vs_2022']].reset_index().sort_values('increase_2023_vs_2022', ascending=False)

# Label EMR for highlighting
region_compare['group'] = np.where(region_compare['region'] == 'EMR', 'EMR', 'Other Regions')

print('Regional measles increase (2023 vs 2022):')
print(region_compare[['region', 'increase_2023_vs_2022']])

fig = px.bar(
    region_compare,
    x='region',
    y='increase_2023_vs_2022',
    color='group',
    color_discrete_map={'EMR': '#d62728', 'Other Regions': '#1f77b4'},
    title='EMR vs Other Regions: Measles Increase (2023 vs 2022)',
    labels={'increase_2023_vs_2022': 'Increase in Measles Cases', 'region': 'Region'},
    text='increase_2023_vs_2022'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=550, xaxis=dict(type='category'), yaxis_tickformat=',', hovermode='x unified')
fig.show()

Regional measles increase (2023 vs 2022):
year region  increase_2023_vs_2022
3       EUR                60043.0
4      SEAR                41340.0
2       EMR                33390.0
0       AFR                 8521.0
5       WPR                 4351.0
1       AMR                  -96.0


In [31]:
# Countries with high measles burden but low lab confirmation
import pandas as pd
import plotly.express as px

# Use in-memory cleaned data if available; otherwise load from disk
try:
    df = yearly_measles.copy()
except NameError:
    df = pd.read_csv('../data/cleaned/cases_year_clean.csv')

country_confirm = (
    df.groupby('country', as_index=False)
    .agg(
        measles_total=('measles_total', 'sum'),
        measles_lab_confirmed=('measles_lab_confirmed', 'sum')
    )
)

# Avoid divide-by-zero and compute confirmation rate
country_confirm['confirmation_rate_pct'] = (
    country_confirm['measles_lab_confirmed']
    .div(country_confirm['measles_total'].replace(0, pd.NA))
    .mul(100)
)

# Define "high cases" by 75th percentile and "low confirmation" as < 30%
high_case_cutoff = country_confirm['measles_total'].quantile(0.75)
low_confirmation_cutoff = 30

high_low = country_confirm[
    (country_confirm['measles_total'] >= high_case_cutoff) &
    (country_confirm['confirmation_rate_pct'] < low_confirmation_cutoff)
].sort_values(['measles_total', 'confirmation_rate_pct'], ascending=[False, True])

print(f"High-case cutoff (75th percentile): {int(high_case_cutoff):,} cases")
print(f"Low-confirmation cutoff: < {low_confirmation_cutoff}%")
print(f"Countries flagged: {len(high_low)}")
print(high_low[['country', 'measles_total', 'measles_lab_confirmed', 'confirmation_rate_pct']].head(20))

# Interactive bubble chart
fig = px.scatter(
    country_confirm,
    x='measles_total',
    y='confirmation_rate_pct',
    hover_name='country',
    size='measles_total',
    size_max=45,
    opacity=0.7,
    title='Countries with High Cases but Low Lab Confirmation',
    labels={
        'measles_total': 'Total Measles Cases',
        'confirmation_rate_pct': 'Lab Confirmation Rate (%)'
    }
)

# Threshold reference lines
fig.add_vline(x=high_case_cutoff, line_dash='dash', line_color='orange',
              annotation_text='High-case threshold', annotation_position='top right')
fig.add_hline(y=low_confirmation_cutoff, line_dash='dash', line_color='red',
              annotation_text='Low-confirmation threshold', annotation_position='bottom right')

# Highlight flagged countries
if not high_low.empty:
    fig.add_scatter(
        x=high_low['measles_total'],
        y=high_low['confirmation_rate_pct'],
        mode='markers+text',
        text=high_low['country'],
        textposition='top center',
        marker=dict(color='crimson', size=12, symbol='diamond'),
        name='Flagged countries'
    )

fig.update_layout(height=650, yaxis_range=[0, 105], xaxis_tickformat=',', hovermode='closest')
fig.show()

High-case cutoff (75th percentile): 7,808 cases
Low-confirmation cutoff: < 30%
Countries flagged: 21
                              country  measles_total  measles_lab_confirmed  \
78                              India         474525                  78576   
101                        Madagascar         225947                   1745   
124                           Nigeria         210073                  29000   
135                       Philippines         148930                  28448   
190                             Yemen         147829                   9428   
180                           Ukraine         136311                  21573   
59                           Ethiopia         117825                  17793   
47   Democratic Republic of the Congo          84184                  22059   
79                          Indonesia          83286                  22029   
159                           Somalia          68014                   4379   
81                            

In [32]:
# Countries driving the 2019 EUR measles spike (vs 2018)
import pandas as pd
import plotly.express as px

# Resolve source data robustly
try:
    source_df = yearly_measles.copy()
except NameError:
    try:
        source_df = df.copy()
    except NameError:
        source_df = pd.read_csv('../data/cleaned/cases_year_clean.csv')

# Keep EUR records (handles both mapped and unmapped codes)
eur_df = source_df[source_df['region'].isin(['EUR', 'EURO'])].copy()

country_year_eur = (
    eur_df.groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

pivot_eur = country_year_eur.pivot(index='country', columns='year', values='measles_total').fillna(0)
pivot_eur['increase_2019_vs_2018'] = pivot_eur.get(2019, 0) - pivot_eur.get(2018, 0)

eur_spike_drivers = (
    pivot_eur[pivot_eur['increase_2019_vs_2018'] > 0][['increase_2019_vs_2018']]
    .sort_values('increase_2019_vs_2018', ascending=False)
)

eur_global_spike = eur_spike_drivers['increase_2019_vs_2018'].sum()
eur_top_n = 15
eur_top_drivers = eur_spike_drivers.head(eur_top_n).copy()
eur_top_drivers['share_of_eur_spike_pct'] = (
    eur_top_drivers['increase_2019_vs_2018'] / eur_global_spike * 100
).round(2)

print('Countries driving the EUR 2019 spike (increase from 2018 to 2019):')
print(eur_top_drivers)
print(f"\nTotal EUR increase (2019 vs 2018): {int(eur_global_spike):,}")
print(
    f"Top {eur_top_n} countries explain "
    f"{eur_top_drivers['share_of_eur_spike_pct'].sum():.2f}% of the EUR increase."
)

# Interactive chart
eur_plot = eur_top_drivers.sort_values('increase_2019_vs_2018', ascending=True).reset_index()
fig = px.bar(
    eur_plot,
    y='country',
    x='increase_2019_vs_2018',
    orientation='h',
    color='increase_2019_vs_2018',
    color_continuous_scale='Blues',
    title='Top EUR Countries Driving the 2019 Measles Spike (vs 2018)',
    labels={
        'increase_2019_vs_2018': 'Increase in Measles Cases',
        'country': 'Country'
    },
    hover_data={'share_of_eur_spike_pct': True}
)
fig.update_layout(height=700, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig.show()

Countries driving the EUR 2019 spike (increase from 2018 to 2019):
year                    increase_2019_vs_2018  share_of_eur_spike_pct
country                                                              
Kazakhstan                            12750.0                   37.99
Ukraine                                4114.0                   12.26
Türkiye                                2176.0                    6.48
Russian Federation                     1881.0                    5.60
North Macedonia                        1820.0                    5.42
Uzbekistan                             1722.0                    5.13
Georgia                                1718.0                    5.12
Poland                                 1367.0                    4.07
Bosnia and Herzegovina                 1312.0                    3.91
Bulgaria                               1234.0                    3.68
Kyrgyzstan                             1199.0                    3.57
Lithuania              

In [33]:
# Investigate if Africa (AFR) is plateauing at high measles levels
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Filter for African Region (AFR)
afr_data = yearly_measles[yearly_measles['region'] == 'AFR'].copy()

# Aggregate by year: total cases and median incidence rate
afr_yearly = afr_data.groupby('year', as_index=False).agg({
    'measles_total': 'sum',
    'measles_incidence_rate_per_1000000_total_population': ['median', 'mean'],
    'country': 'count'
}).reset_index(drop=True)

afr_yearly.columns = ['year', 'total_cases', 'median_incidence', 'mean_incidence', 'num_countries']

print("=" * 80)
print("AFRICA (AFR) MEASLES TREND ANALYSIS")
print("=" * 80)
print("\nAnnual Summary:")
print(afr_yearly.to_string(index=False))

# Calculate year-over-year change
afr_yearly['yoy_change'] = afr_yearly['total_cases'].diff()
afr_yearly['yoy_pct_change'] = ((afr_yearly['total_cases'] / afr_yearly['total_cases'].shift(1)) - 1) * 100

# Identify if plateauing: look for periods with small YoY changes (< 10% absolute change)
afr_yearly['is_stable'] = abs(afr_yearly['yoy_pct_change']) < 10

print("\n" + "=" * 80)
print("YEAR-OVER-YEAR ANALYSIS (Identifying Plateaus):")
print("=" * 80)
print(afr_yearly[['year', 'total_cases', 'yoy_change', 'yoy_pct_change', 'is_stable']].to_string(index=False))

# Find recent years (last 5 years) and assess if high plateau
recent_years = afr_yearly.tail(5)
recent_avg = recent_years['total_cases'].mean()
overall_avg = afr_yearly['total_cases'].mean()
recent_is_high = recent_avg > overall_avg * 1.25  # 25% above average

print("\n" + "=" * 80)
print("PLATEAU ASSESSMENT:")
print("=" * 80)
print(f"Historical average cases/year (all years): {overall_avg:,.0f}")
print(f"Recent years average (last 5 years): {recent_avg:,.0f}")
print(f"Recent years are {'HIGH' if recent_is_high else 'NORMAL'} compared to history")

# Check for stability in recent period
recent_stability_pct = recent_years['is_stable'].sum() / len(recent_years) * 100
print(f"Stability in last 5 years: {recent_stability_pct:.0f}% of years had <10% YoY change")

if recent_is_high and recent_stability_pct >= 60:
    print("✓ CONCLUSION: Africa appears to be PLATEAUING at HIGH levels of measles")
elif recent_is_high and recent_stability_pct < 60:
    print("⚠ CONCLUSION: Africa has high cases but with volatility (not stable plateau)")
elif recent_stability_pct >= 60:
    print("✓ CONCLUSION: Africa shows stability but at normal/low levels")
else:
    print("• CONCLUSION: Africa shows variable trends without clear plateau pattern")

# Interactive visualization with subplots
fig = make_subplots(rows=2, cols=1, 
                     subplot_titles=("Total Measles Cases Over Time", "Year-over-Year Percent Change"),
                     specs=[[{"secondary_y": False}], [{"secondary_y": False}]])

# Plot 1: Total cases with trend overlay
fig.add_trace(
    go.Scatter(x=afr_yearly['year'], y=afr_yearly['total_cases'],
               mode='lines+markers', name='Total Cases', 
               line=dict(color='#d62728', width=3), marker=dict(size=8)),
    row=1, col=1
)

# Add rolling average (3-year) to show trend
rolling_avg = afr_yearly['total_cases'].rolling(window=3, center=True).mean()
fig.add_trace(
    go.Scatter(x=afr_yearly['year'], y=rolling_avg,
               mode='lines', name='3-Year Rolling Avg',
               line=dict(color='#ff7f0e', width=2, dash='dash')),
    row=1, col=1
)

# Plot 2: YoY percent change
colors = ['#2ca02c' if x else '#1f77b4' for x in afr_yearly['is_stable']]
fig.add_trace(
    go.Bar(x=afr_yearly['year'], y=afr_yearly['yoy_pct_change'],
           name='YoY % Change', marker=dict(color=colors),
           text=[f"{x:.1f}%" if pd.notna(x) else "" for x in afr_yearly['yoy_pct_change']],
           textposition='outside'),
    row=2, col=1
)

# Add stability threshold lines
fig.add_hline(y=10, line_dash="dash", line_color="green", 
              annotation_text="Stability threshold (±10%)", annotation_position="right",
              row=2, col=1)
fig.add_hline(y=-10, line_dash="dash", line_color="green", row=2, col=1)

fig.update_xaxes(title_text="Year", row=1, col=1)
fig.update_xaxes(title_text="Year", row=2, col=1)
fig.update_yaxes(title_text="Total Cases", row=1, col=1)
fig.update_yaxes(title_text="YoY % Change", row=2, col=1)

fig.update_layout(height=800, title_text="Africa (AFR): Is Measles Plateauing at High Levels?",
                  hovermode='x unified', showlegend=True)
fig.show()

AFRICA (AFR) MEASLES TREND ANALYSIS

Annual Summary:
 year  total_cases  median_incidence  mean_incidence  num_countries
 2012        24791             9.510       20.455814             43
 2013        77073             6.045       30.735238             42
 2014        42957             8.630       38.410698             43
 2015        46103             5.270       54.394186             43
 2016        36753             8.380       60.994186             43
 2017        24845             5.960       24.253864             44
 2018        42167            11.640       38.407674             43
 2019       289766            17.840      215.768444             45
 2020        46747            11.240       51.727381             42
 2021        26492             9.670       17.273415             41
 2022        64922            25.990       72.024773             44
 2023        73443            22.875       73.521364             44
 2024        86127            24.420       64.276977           

In [34]:
# African countries driving the 2023 measles spike (vs 2022)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Filter for African Region (AFR)
afr_data = yearly_measles[yearly_measles['region'] == 'AFR'].copy()

# Aggregate by country and year
afr_country_year = (
    afr_data
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

# Pivot to calculate year-over-year change
afr_pivot = afr_country_year.pivot(index='country', columns='year', values='measles_total').fillna(0)
afr_pivot['increase_2023_vs_2022'] = afr_pivot.get(2023, 0) - afr_pivot.get(2022, 0)

# Filter for countries with positive increases
afr_spike_drivers = (
    afr_pivot[afr_pivot['increase_2023_vs_2022'] > 0][['increase_2023_vs_2022']]
    .sort_values('increase_2023_vs_2022', ascending=False)
)

afr_global_spike = afr_spike_drivers['increase_2023_vs_2022'].sum()
afr_top_n = 15
afr_top_drivers = afr_spike_drivers.head(afr_top_n).copy()
afr_top_drivers['share_of_afr_spike_pct'] = (
    afr_top_drivers['increase_2023_vs_2022'] / afr_global_spike * 100
).round(2)

print("=" * 80)
print("AFRICAN COUNTRIES DRIVING THE 2023 MEASLES SPIKE (vs 2022)")
print("=" * 80)
print("\nTop Countries Contributing to the Spike:")
print(afr_top_drivers)
print(f"\nTotal AFR increase (2023 vs 2022): {int(afr_global_spike):,}")
print(f"Top {afr_top_n} countries explain {afr_top_drivers['share_of_afr_spike_pct'].sum():.2f}% of the AFR spike")

# Identify if spike was concentrated or distributed
top_5_share = afr_top_drivers.head(5)['share_of_afr_spike_pct'].sum()
print(f"\nConcentration: Top 5 countries account for {top_5_share:.1f}% of the spike")

# Interactive horizontal bar chart
afr_plot = afr_top_drivers.sort_values('increase_2023_vs_2022', ascending=True).reset_index()
fig = px.bar(
    afr_plot,
    y='country',
    x='increase_2023_vs_2022',
    orientation='h',
    color='increase_2023_vs_2022',
    color_continuous_scale='Greens',
    title='Top African Countries Driving the 2023 Measles Spike (vs 2022)',
    labels={
        'increase_2023_vs_2022': 'Increase in Measles Cases',
        'country': 'Country'
    },
    hover_data={'share_of_afr_spike_pct': True}
)
fig.update_layout(height=650, showlegend=False, hovermode='y unified')
fig.show()

# Additional insight: Compare 2022 baseline vs 2023 for top drivers
print("\n" + "=" * 80)
print("BASELINE COMPARISON FOR TOP DRIVERS")
print("=" * 80)
top_drivers_baseline = afr_pivot.loc[afr_top_drivers.index, [2022, 2023, 'increase_2023_vs_2022']].reset_index()
top_drivers_baseline.columns = ['country', 'cases_2022', 'cases_2023', 'increase']
top_drivers_baseline['pct_increase'] = ((top_drivers_baseline['cases_2023'] / top_drivers_baseline['cases_2022'].replace(0, pd.NA)) - 1) * 100
print(top_drivers_baseline.head(10).to_string(index=False))

AFRICAN COUNTRIES DRIVING THE 2023 MEASLES SPIKE (vs 2022)

Top Countries Contributing to the Spike:
year                              increase_2023_vs_2022  \
country                                                   
Ethiopia                                         8289.0   
Democratic Republic of the Congo                 6336.0   
Cameroon                                         3114.0   
Ghana                                            2037.0   
Burundi                                          1472.0   
Central African Republic                         1377.0   
Chad                                             1155.0   
Kenya                                             717.0   
United Republic of Tanzania                       716.0   
South Africa                                      633.0   
Burkina Faso                                      559.0   
Mauritania                                        322.0   
Equatorial Guinea                                 263.0   
Uganda        


BASELINE COMPARISON FOR TOP DRIVERS
                         country  cases_2022  cases_2023  increase  pct_increase
                        Ethiopia      8216.0     16505.0    8289.0    100.888510
Democratic Republic of the Congo      4365.0     10701.0    6336.0    145.154639
                        Cameroon      3061.0      6175.0    3114.0    101.731460
                           Ghana       242.0      2279.0    2037.0    841.735537
                         Burundi       222.0      1694.0    1472.0    663.063063
        Central African Republic       157.0      1534.0    1377.0    877.070064
                            Chad       316.0      1471.0    1155.0    365.506329
                           Kenya       140.0       857.0     717.0    512.142857
     United Republic of Tanzania       520.0      1236.0     716.0    137.692308
                    South Africa       255.0       888.0     633.0    248.235294


In [35]:
# African countries driving the 2019 measles spike (vs 2018)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Filter for African Region (AFR)
afr_data_2019 = yearly_measles[yearly_measles['region'] == 'AFR'].copy()

# Aggregate by country and year
afr_country_year_2019 = (
    afr_data_2019
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

# Pivot to calculate year-over-year change
afr_pivot_2019 = afr_country_year_2019.pivot(index='country', columns='year', values='measles_total').fillna(0)
afr_pivot_2019['increase_2019_vs_2018'] = afr_pivot_2019.get(2019, 0) - afr_pivot_2019.get(2018, 0)

# Filter for countries with positive increases
afr_spike_drivers_2019 = (
    afr_pivot_2019[afr_pivot_2019['increase_2019_vs_2018'] > 0][['increase_2019_vs_2018']]
    .sort_values('increase_2019_vs_2018', ascending=False)
)

afr_global_spike_2019 = afr_spike_drivers_2019['increase_2019_vs_2018'].sum()
afr_top_n_2019 = 15
afr_top_drivers_2019 = afr_spike_drivers_2019.head(afr_top_n_2019).copy()
afr_top_drivers_2019['share_of_afr_spike_pct'] = (
    afr_top_drivers_2019['increase_2019_vs_2018'] / afr_global_spike_2019 * 100
).round(2)

print("=" * 80)
print("AFRICAN COUNTRIES DRIVING THE 2019 MEASLES SPIKE (vs 2018)")
print("=" * 80)
print("\nTop Countries Contributing to the Spike:")
print(afr_top_drivers_2019)
print(f"\nTotal AFR increase (2019 vs 2018): {int(afr_global_spike_2019):,}")
print(f"Top {afr_top_n_2019} countries explain {afr_top_drivers_2019['share_of_afr_spike_pct'].sum():.2f}% of the AFR spike")

# Identify if spike was concentrated or distributed
top_5_share_2019 = afr_top_drivers_2019.head(5)['share_of_afr_spike_pct'].sum()
print(f"\nConcentration: Top 5 countries account for {top_5_share_2019:.1f}% of the spike")

# Interactive horizontal bar chart
afr_plot_2019 = afr_top_drivers_2019.sort_values('increase_2019_vs_2018', ascending=True).reset_index()
fig = px.bar(
    afr_plot_2019,
    y='country',
    x='increase_2019_vs_2018',
    orientation='h',
    color='increase_2019_vs_2018',
    color_continuous_scale='Purples',
    title='Top African Countries Driving the 2019 Measles Spike (vs 2018)',
    labels={
        'increase_2019_vs_2018': 'Increase in Measles Cases',
        'country': 'Country'
    },
    hover_data={'share_of_afr_spike_pct': True}
)
fig.update_layout(height=650, showlegend=False, hovermode='y unified')
fig.show()

# Additional insight: Compare 2018 baseline vs 2019 for top drivers
print("\n" + "=" * 80)
print("BASELINE COMPARISON FOR TOP DRIVERS")
print("=" * 80)
top_drivers_baseline_2019 = afr_pivot_2019.loc[afr_top_drivers_2019.index, [2018, 2019, 'increase_2019_vs_2018']].reset_index()
top_drivers_baseline_2019.columns = ['country', 'cases_2018', 'cases_2019', 'increase']
top_drivers_baseline_2019['pct_increase'] = ((top_drivers_baseline_2019['cases_2019'] / top_drivers_baseline_2019['cases_2018'].replace(0, pd.NA)) - 1) * 100
print(top_drivers_baseline_2019.head(10).to_string(index=False))

AFRICAN COUNTRIES DRIVING THE 2019 MEASLES SPIKE (vs 2018)

Top Countries Contributing to the Spike:
year                              increase_2019_vs_2018  \
country                                                   
Madagascar                                     201239.0   
Nigeria                                         21466.0   
Democratic Republic of the Congo                12975.0   
Angola                                           2689.0   
Algeria                                          2298.0   
Ethiopia                                         2259.0   
Cameroon                                         2018.0   
Central African Republic                         1791.0   
South Sudan                                      1345.0   
Niger                                            1123.0   
Chad                                              890.0   
Mozambique                                        691.0   
Guinea                                            690.0   
Senegal       


BASELINE COMPARISON FOR TOP DRIVERS
                         country  cases_2018  cases_2019  increase  pct_increase
                      Madagascar     12052.0    213291.0  201239.0   1669.756057
                         Nigeria      6836.0     28302.0   21466.0    314.014043
Democratic Republic of the Congo      5492.0     18467.0   12975.0    236.252731
                          Angola       291.0      2980.0    2689.0    924.054983
                         Algeria       287.0      2585.0    2298.0    800.696864
                        Ethiopia      1351.0      3610.0    2259.0    167.209474
                        Cameroon       831.0      2849.0    2018.0    242.839952
        Central African Republic        24.0      1815.0    1791.0   7462.500000
                     South Sudan       127.0      1472.0    1345.0   1059.055118
                           Niger      1445.0      2568.0    1123.0     77.716263


In [36]:
# Watch list: countries with >500% increase but still under 5,000 cases (latest year vs previous year)
import pandas as pd
import plotly.express as px

country_year = (
    yearly_measles
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

latest_year = int(country_year['year'].max())
prev_year = latest_year - 1

year_pair = country_year[country_year['year'].isin([prev_year, latest_year])].copy()
pivot_watch = year_pair.pivot(index='country', columns='year', values='measles_total').fillna(0)

prev_col = pivot_watch.get(prev_year, pd.Series(0, index=pivot_watch.index))
latest_col = pivot_watch.get(latest_year, pd.Series(0, index=pivot_watch.index))

# Percent increase with safe divide-by-zero handling:
# if previous year is 0 and latest > 0, treat as infinite growth.
pct_increase = ((latest_col - prev_col) / prev_col.replace(0, pd.NA)) * 100
pct_increase = pct_increase.mask((prev_col == 0) & (latest_col > 0), float('inf'))

watch_list = pd.DataFrame({
    'cases_prev_year': prev_col,
    'cases_latest_year': latest_col,
    'pct_increase': pct_increase,
    'absolute_increase': latest_col - prev_col
}).reset_index()

watch_list_flagged = (
    watch_list[
        (watch_list['pct_increase'] > 500) &
        (watch_list['cases_latest_year'] < 5000)
    ]
    .sort_values(['pct_increase', 'cases_latest_year'], ascending=[False, False])
)

print('=' * 80)
print(f'WATCH LIST: >500% INCREASE BUT <5,000 CASES ({prev_year} -> {latest_year})')
print('=' * 80)
print(f'Countries flagged: {len(watch_list_flagged)}')

if watch_list_flagged.empty:
    print('No countries meet the watch-list criteria for this year pair.')
else:
    display_cols = ['country', 'cases_prev_year', 'cases_latest_year', 'absolute_increase', 'pct_increase']
    print(watch_list_flagged[display_cols].to_string(index=False))

    # Plot only finite percentage increases for readability
    plot_df = watch_list_flagged[watch_list_flagged['pct_increase'] != float('inf')].copy()

    if not plot_df.empty:
        fig = px.bar(
            plot_df.head(20).sort_values('pct_increase', ascending=True),
            y='country',
            x='pct_increase',
            orientation='h',
            color='cases_latest_year',
            color_continuous_scale='YlOrRd',
            title=f'Watch List Countries by % Increase ({prev_year} -> {latest_year})',
            labels={
                'pct_increase': 'Percent Increase (%)',
                'country': 'Country',
                'cases_latest_year': f'Cases in {latest_year}'
            },
            hover_data={
                'cases_prev_year': True,
                'cases_latest_year': True,
                'absolute_increase': True
            }
        )
        fig.update_layout(height=650, showlegend=False, hovermode='y unified')
        fig.show()
    else:
        print('All flagged countries had zero baseline in previous year (infinite % increase), so no finite % bar chart was generated.')

WATCH LIST: >500% INCREASE BUT <5,000 CASES (2024 -> 2025)
Countries flagged: 8
                         country  cases_prev_year  cases_latest_year  absolute_increase  pct_increase
                      Tajikistan              0.0             1416.0             1416.0           inf
                          Belize              0.0                7.0                7.0           inf
                          Bhutan              0.0                4.0                4.0           inf
                      Costa Rica              0.0                1.0                1.0           inf
                          Mexico              7.0             1926.0             1919.0  27414.285714
Lao People's Democratic Republic              1.0              114.0              113.0       11300.0
                        Mongolia             12.0              382.0              370.0   3083.333333
                          Canada            146.0             2197.0             2051.0   1404.794521


In [37]:
# What fraction of 2025 expected reports have actually come in?
total_countries = yearly_measles[yearly_measles["year"] == 2024]["iso3"].nunique()
reporting_2025 = yearly_measles[yearly_measles["year"] == 2025]["iso3"].nunique()
print(f"2025 reporting coverage: {reporting_2025/total_countries:.1%}")

2025 reporting coverage: 91.3%


In [38]:
# Africa watch list: countries with >500% increase but still under 5,000 cases (latest year vs previous year)
import pandas as pd
import plotly.express as px

afr_watch_source = yearly_measles[yearly_measles['region'] == 'AFR'].copy()

afr_country_year_watch = (
    afr_watch_source
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

latest_year_afr = int(afr_country_year_watch['year'].max())
prev_year_afr = latest_year_afr - 1

afr_year_pair = afr_country_year_watch[
    afr_country_year_watch['year'].isin([prev_year_afr, latest_year_afr])
].copy()

afr_pivot_watch = afr_year_pair.pivot(index='country', columns='year', values='measles_total').fillna(0)

prev_afr = afr_pivot_watch.get(prev_year_afr, pd.Series(0, index=afr_pivot_watch.index))
latest_afr = afr_pivot_watch.get(latest_year_afr, pd.Series(0, index=afr_pivot_watch.index))

# Safe percent increase calculation.
# If previous year is 0 and latest year > 0, mark as infinite growth.
afr_pct_increase = ((latest_afr - prev_afr) / prev_afr.replace(0, pd.NA)) * 100
afr_pct_increase = afr_pct_increase.mask((prev_afr == 0) & (latest_afr > 0), float('inf'))

afr_watch = pd.DataFrame({
    'cases_prev_year': prev_afr,
    'cases_latest_year': latest_afr,
    'pct_increase': afr_pct_increase,
    'absolute_increase': latest_afr - prev_afr
}).reset_index()

afr_watch_flagged = (
    afr_watch[
        (afr_watch['pct_increase'] > 500) &
        (afr_watch['cases_latest_year'] < 5000)
    ]
    .sort_values(['pct_increase', 'cases_latest_year'], ascending=[False, False])
)

print('=' * 80)
print(f'AFRICA WATCH LIST: >500% INCREASE BUT <5,000 CASES ({prev_year_afr} -> {latest_year_afr})')
print('=' * 80)
print(f'Countries flagged: {len(afr_watch_flagged)}')

if afr_watch_flagged.empty:
    print('No African countries meet the watch-list criteria for this year pair.')
else:
    cols = ['country', 'cases_prev_year', 'cases_latest_year', 'absolute_increase', 'pct_increase']
    print(afr_watch_flagged[cols].to_string(index=False))

    # Plot finite percentages only for readability.
    afr_plot_df = afr_watch_flagged[afr_watch_flagged['pct_increase'] != float('inf')].copy()

    if not afr_plot_df.empty:
        fig = px.bar(
            afr_plot_df.head(20).sort_values('pct_increase', ascending=True),
            y='country',
            x='pct_increase',
            orientation='h',
            color='cases_latest_year',
            color_continuous_scale='YlGnBu',
            title=f'Africa Watch List by Percent Increase ({prev_year_afr} -> {latest_year_afr})',
            labels={
                'pct_increase': 'Percent Increase (%)',
                'country': 'Country',
                'cases_latest_year': f'Cases in {latest_year_afr}'
            },
            hover_data={
                'cases_prev_year': True,
                'cases_latest_year': True,
                'absolute_increase': True
            }
        )
        fig.update_layout(height=650, showlegend=False, hovermode='y unified')
        fig.show()
    else:
        print('All flagged African countries had zero baseline in the previous year, so no finite-percent bar chart was generated.')

AFRICA WATCH LIST: >500% INCREASE BUT <5,000 CASES (2024 -> 2025)
Countries flagged: 0
No African countries meet the watch-list criteria for this year pair.


In [39]:
# Confirm whether African measles cases are declining or stable in 2025
import pandas as pd
import plotly.graph_objects as go

afr_annual = (
    yearly_measles[yearly_measles['region'] == 'AFR']
    .groupby('year', as_index=False)['measles_total']
    .sum()
    .sort_values('year')
)

if 2025 not in afr_annual['year'].values or 2024 not in afr_annual['year'].values:
    print('Required years (2024 and 2025) are not both available for AFR trend confirmation.')
else:
    afr_annual['yoy_pct'] = afr_annual['measles_total'].pct_change() * 100

    cases_2024 = float(afr_annual.loc[afr_annual['year'] == 2024, 'measles_total'].iloc[0])
    cases_2025 = float(afr_annual.loc[afr_annual['year'] == 2025, 'measles_total'].iloc[0])
    yoy_2025 = float(afr_annual.loc[afr_annual['year'] == 2025, 'yoy_pct'].iloc[0])

    # Context baseline: average of previous 3 years (2022-2024 when available)
    baseline_years = [y for y in [2022, 2023, 2024] if y in afr_annual['year'].values]
    baseline_avg = float(afr_annual.loc[afr_annual['year'].isin(baseline_years), 'measles_total'].mean()) if baseline_years else float('nan')

    # Decision rules
    # declining: >=10% drop YoY
    # stable: within +/-5% YoY
    # otherwise: increasing/volatile
    if yoy_2025 <= -10:
        verdict = 'DECLINING'
    elif -5 <= yoy_2025 <= 5:
        verdict = 'STABLE'
    else:
        verdict = 'NOT STABLE (increasing or volatile)'

    print('=' * 80)
    print('AFRICA 2025 STATUS CHECK: DECLINING OR STABLE?')
    print('=' * 80)
    print(f'AFR cases in 2024: {cases_2024:,.0f}')
    print(f'AFR cases in 2025: {cases_2025:,.0f}')
    print(f'YoY change (2025 vs 2024): {yoy_2025:.2f}%')
    if pd.notna(baseline_avg):
        print(f'Baseline average ({min(baseline_years)}-{max(baseline_years)}): {baseline_avg:,.0f}')
        print(f'2025 vs baseline: {((cases_2025 / baseline_avg) - 1) * 100:.2f}%')
    print(f'VERDICT: {verdict}')

    # Visual check focused on recent years
    recent = afr_annual[afr_annual['year'] >= 2021].copy()
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=recent['year'],
            y=recent['measles_total'],
            mode='lines+markers+text',
            text=[f"{v:,.0f}" for v in recent['measles_total']],
            textposition='top center',
            name='AFR total measles cases',
            line=dict(color='#1f77b4', width=3),
            marker=dict(size=8)
        )
    )

    # Highlight 2025 point
    fig.add_trace(
        go.Scatter(
            x=[2025],
            y=[cases_2025],
            mode='markers+text',
            text=[verdict],
            textposition='bottom center',
            name='2025 classification',
            marker=dict(color='#d62728', size=12, symbol='diamond')
        )
    )

    fig.update_layout(
        title='Africa Measles Cases: Is 2025 Declining or Stable?',
        xaxis_title='Year',
        yaxis_title='Total Measles Cases',
        hovermode='x unified',
        height=550
    )
    fig.show()

AFRICA 2025 STATUS CHECK: DECLINING OR STABLE?
AFR cases in 2024: 86,127
AFR cases in 2025: 18,832
YoY change (2025 vs 2024): -78.13%
Baseline average (2022-2024): 74,831
2025 vs baseline: -74.83%
VERDICT: DECLINING


In [40]:
# Check which specific AFR countries are missing in 2025
afr_2024_countries = set(yearly_measles[(yearly_measles["region"] == "AFR") & (yearly_measles["year"] == 2024)]["iso3"])
afr_2025_countries = set(yearly_measles[(yearly_measles["region"] == "AFR") & (yearly_measles["year"] == 2025)]["iso3"])
missing = afr_2024_countries - afr_2025_countries
print(f"Missing AFR countries in 2025: {len(missing)}")
print(missing)

# Are any of your high-burden countries missing?
high_burden = ["ETH", "COD", "CMR", "CAF", "MDG"]
print("High burden countries missing:", [c for c in high_burden if c in missing])

Missing AFR countries in 2025: 2
{'AGO', 'ERI'}
High burden countries missing: []


In [42]:
# Is the 2025 AFR decline broad-based or mostly one-country recovery?
import pandas as pd
import plotly.express as px

afr_country_year_24_25 = (
    yearly_measles[yearly_measles['region'] == 'AFR']
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

required_years = {2024, 2025}
available_years = set(afr_country_year_24_25['year'].unique())

if not required_years.issubset(available_years):
    print('Cannot run decomposition: both 2024 and 2025 are required for AFR.')
else:
    afr_pivot_24_25 = (
        afr_country_year_24_25[afr_country_year_24_25['year'].isin([2024, 2025])]
        .pivot(index='country', columns='year', values='measles_total')
        .fillna(0)
    )

    # Handle year columns whether they are ints (2024) or strings ('2024')
    col_2024 = 2024 if 2024 in afr_pivot_24_25.columns else '2024'
    col_2025 = 2025 if 2025 in afr_pivot_24_25.columns else '2025'

    if col_2024 not in afr_pivot_24_25.columns or col_2025 not in afr_pivot_24_25.columns:
        print('Cannot run decomposition: 2024 and 2025 columns are missing after pivot.')
    else:
        afr_pivot_24_25['delta_2025_vs_2024'] = afr_pivot_24_25[col_2025] - afr_pivot_24_25[col_2024]

        declines = afr_pivot_24_25[afr_pivot_24_25['delta_2025_vs_2024'] < 0].copy()
        increases = afr_pivot_24_25[afr_pivot_24_25['delta_2025_vs_2024'] > 0].copy()

        total_change = afr_pivot_24_25['delta_2025_vs_2024'].sum()  # likely negative if decline
        total_decline_abs = -declines['delta_2025_vs_2024'].sum() if not declines.empty else 0.0

        print('=' * 90)
        print('AFR 2025 DECOMPOSITION: BROAD-BASED DECLINE VS ONE-COUNTRY RECOVERY')
        print('=' * 90)
        print(f'Total AFR change (2025 vs 2024): {int(total_change):,}')
        print(f'Number of countries declining: {len(declines)}')
        print(f'Number of countries increasing: {len(increases)}')

        if declines.empty:
            print('No country-level declines found, so AFR decline is not broad-based.')
        else:
            top_decliner = declines.sort_values('delta_2025_vs_2024').head(1)
            top_decliner_country = top_decliner.index[0]
            top_decliner_abs = int(-top_decliner['delta_2025_vs_2024'].iloc[0])

            top_decliners = declines.sort_values('delta_2025_vs_2024').copy()
            top_decliners['decline_abs'] = -top_decliners['delta_2025_vs_2024']
            top_decliners['share_of_total_decline_pct'] = (
                top_decliners['decline_abs'] / total_decline_abs * 100
            ).round(2)

            # Rename year columns to stable string names for downstream indexing/plotting
            top_decliners = top_decliners.rename(columns={col_2024: 'cases_2024', col_2025: 'cases_2025'})

            top_country_share = float(top_decliners['share_of_total_decline_pct'].iloc[0]) if total_decline_abs > 0 else 0.0
            top_5_share = float(top_decliners['share_of_total_decline_pct'].head(5).sum()) if total_decline_abs > 0 else 0.0

            print(f'Largest single-country decline: {top_decliner_country} ({top_decliner_abs:,} cases)')
            print(f'Share of total AFR decline from largest country: {top_country_share:.2f}%')
            print(f'Share of total AFR decline from top 5 declining countries: {top_5_share:.2f}%')

            # Classification rule of thumb
            # One-country-led if largest country explains >=50% of decline.
            if top_country_share >= 50:
                classification = 'Mostly one-country recovery'
            else:
                classification = 'Broad-based decline across multiple countries'

            print(f'Classification: {classification}')

            # Show top contributing decliners
            show_cols = ['cases_2024', 'cases_2025', 'decline_abs', 'share_of_total_decline_pct']
            print('\nTop declining countries driving 2025 AFR drop:')
            print(top_decliners[show_cols].head(15).to_string())

            # Bar chart for contribution shares
            plot_df = top_decliners.reset_index().head(15).sort_values('share_of_total_decline_pct', ascending=True)
            fig = px.bar(
                plot_df,
                y='country',
                x='share_of_total_decline_pct',
                orientation='h',
                color='decline_abs',
                color_continuous_scale='Teal',
                title='Who Drove the AFR 2025 Decline? Share of Total Country-Level Decline',
                labels={
                    'share_of_total_decline_pct': 'Share of AFR Decline (%)',
                    'country': 'Country',
                    'decline_abs': 'Absolute Decline (cases)'
                },
                hover_data={'cases_2024': True, 'cases_2025': True, 'decline_abs': True}
            )
            fig.update_layout(height=650, showlegend=False, hovermode='y unified')
            fig.show()

AFR 2025 DECOMPOSITION: BROAD-BASED DECLINE VS ONE-COUNTRY RECOVERY
Total AFR change (2025 vs 2024): -67,295
Number of countries declining: 36
Number of countries increasing: 5
Largest single-country decline: Ethiopia (25,420 cases)
Share of total AFR decline from largest country: 37.18%
Share of total AFR decline from top 5 declining countries: 72.74%
Classification: Broad-based decline across multiple countries

Top declining countries driving 2025 AFR drop:
year                              cases_2024  cases_2025  decline_abs  share_of_total_decline_pct
country                                                                                          
Ethiopia                             30201.0      4781.0      25420.0                       37.18
Nigeria                              10948.0      2504.0       8444.0                       12.35
Côte d'Ivoire                         6629.0       631.0       5998.0                        8.77
Burkina Faso                          7290.0 

In [43]:
# Did South Africa submit a record with 0, or is it just missing?
yearly_measles[(yearly_measles["iso3"] == "ZAF") & (yearly_measles["year"] == 2025)][["country", "year", "measles_total", "surveillance_reported"]]


,country,year,measles_total,surveillance_reported
518,South Africa,2025,0,True


In [44]:
yearly_measles[(yearly_measles["iso3"] == "AGO") & (yearly_measles["year"] == 2025)]

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,...,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population,surveillance_reported,measles_breakdown_missing


In [45]:
# Measles distribution in 2025 (regional and country concentration)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df_2025 = yearly_measles[yearly_measles['year'] == 2025].copy()

if df_2025.empty:
    print('No 2025 records available in yearly_measles.')
else:
    # Regional distribution
    region_2025 = (
        df_2025.groupby('region', as_index=False)['measles_total']
        .sum()
        .sort_values('measles_total', ascending=False)
    )
    total_2025 = region_2025['measles_total'].sum()
    region_2025['share_pct'] = (region_2025['measles_total'] / total_2025 * 100).round(2)

    print('=' * 80)
    print('MEASLES DISTRIBUTION IN 2025: REGIONS')
    print('=' * 80)
    print(region_2025.to_string(index=False))

    fig_region = px.pie(
        region_2025,
        names='region',
        values='measles_total',
        title='2025 Measles Distribution by Region',
        hole=0.4
    )
    fig_region.update_traces(textposition='inside', textinfo='percent+label')
    fig_region.update_layout(height=550)
    fig_region.show()

    # Country concentration (top 20)
    country_2025 = (
        df_2025.groupby('country', as_index=False)['measles_total']
        .sum()
        .sort_values('measles_total', ascending=False)
    )
    country_2025['share_pct'] = (country_2025['measles_total'] / total_2025 * 100).round(2)

    top_n = 20
    top_country_2025 = country_2025.head(top_n).copy()
    top_country_share = top_country_2025['share_pct'].sum()

    print('\n' + '=' * 80)
    print(f'MEASLES DISTRIBUTION IN 2025: TOP {top_n} COUNTRIES')
    print('=' * 80)
    print(top_country_2025.to_string(index=False))
    print(f"\nTop {top_n} countries account for {top_country_share:.2f}% of all 2025 measles cases.")

    fig_country = px.bar(
        top_country_2025.sort_values('measles_total', ascending=True),
        y='country',
        x='measles_total',
        orientation='h',
        color='share_pct',
        color_continuous_scale='Sunset',
        title=f'2025 Measles Distribution: Top {top_n} Countries',
        labels={
            'measles_total': 'Measles Cases (2025)',
            'country': 'Country',
            'share_pct': 'Share of 2025 cases (%)'
        },
        hover_data={'share_pct': True}
    )
    fig_country.update_layout(height=700, hovermode='y unified')
    fig_country.show()

MEASLES DISTRIBUTION IN 2025: REGIONS
region  measles_total  share_pct
   EMR          31052      34.95
   AFR          18832      21.19
   EUR          18551      20.88
  SEAR          10856      12.22
   AMR           5340       6.01
   WPR           4222       4.75



MEASLES DISTRIBUTION IN 2025: TOP 20 COUNTRIES
                         country  measles_total  share_pct
                           Yemen          12791      14.40
                           India           8351       9.40
                        Pakistan           7927       8.92
                     Afghanistan           6739       7.58
                      Kyrgyzstan           5959       6.71
                        Ethiopia           4781       5.38
                         Romania           3603       4.06
                         Nigeria           2504       2.82
                          Canada           2197       2.47
              Russian Federation           1986       2.24
                          Mexico           1926       2.17
                       Indonesia           1841       2.07
                           Niger           1810       2.04
                           Sudan           1692       1.90
Democratic Republic of the Congo           1523       1.71
        

In [46]:
# Confirm Kyrgyzstan's incidence rate
yearly_measles[(yearly_measles["iso3"] == "KGZ") & (yearly_measles["year"] == 2025)][["country", "measles_total", "measles_incidence_rate_per_1000000_total_population"]]


,country,measles_total,measles_incidence_rate_per_1000000_total_population
1555,Kyrgyzstan,5959,1960.46
